In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [4]:
text = open(
    r"C:\Users\Lenovo\Desktop\NanoGPT\Small_Language_Models\data\shakespeare_literature.txt",
    encoding="utf-8"
).read().splitlines()


In [5]:
text = "\n".join(text)

In [6]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [7]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115393


In [8]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [9]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [10]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [11]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [14]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets:
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
----
when input is [53] the target: 59
when input is [53, 59] the target: 6
when input is [53, 59, 6] the target: 1
when input is [53, 59, 6, 1] the target: 58
when input is [53, 59, 6, 1, 58] the target: 56
when input is [53, 59, 6, 1, 58, 56] the target: 47
when input is [53, 59, 6, 1, 58, 56, 47] the target: 40
when input is [53, 59, 6, 1, 58, 56, 47, 40] the target: 59
when input is [49] the target: 43
when input is [49, 43] the target: 43
when input is [49, 43, 43] the target: 54
when input is [49, 43, 43, 54] the target: 1
when input is [49, 43, 43, 54, 1] the target: 47
when input is [49, 43, 

In [15]:
print(xb) # our input to the transformer

tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])


In [16]:
yb.shape

torch.Size([4, 8])

In [27]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # print(B, T, C)
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([256, 65])
tensor(4.6264, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [28]:
m.token_embedding_table.weight

Parameter containing:
tensor([[ 0.1808, -0.0700, -0.3596,  ...,  1.6097, -0.4032, -0.8345],
        [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
        [ 1.3035, -0.4501,  1.3471,  ...,  0.1910, -0.3425,  1.7955],
        ...,
        [ 0.4222, -1.8111, -1.0118,  ...,  0.5462,  0.2788,  0.7280],
        [-0.8109,  0.2410, -0.1139,  ...,  1.4509,  0.1836,  0.3064],
        [-1.4322, -0.2810, -2.2789,  ..., -0.5551,  1.0666,  0.5364]],
       requires_grad=True)

In [29]:
# Example of target with class indices
input1 = torch.randn(3, 7, requires_grad=True)
target = torch.randint(7, (3,), dtype=torch.int64)
loss = F.cross_entropy(input1, target)
loss.backward()

In [30]:
input1

tensor([[-0.4117, -0.8850,  1.7100, -1.6693, -0.0715, -0.8394,  1.3583],
        [-0.0501, -0.7759,  0.9652, -0.2760, -0.4317, -0.1501, -2.6449],
        [-0.1849,  1.1784, -1.0494, -0.3854,  0.0871, -0.2610, -0.2207]],
       requires_grad=True)

In [31]:
target

tensor([6, 2, 0])

In [32]:
loss = F.cross_entropy(input1, target)
loss

tensor(1.4178, grad_fn=<NllLossBackward0>)

In [33]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [34]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


4.609185218811035


In [35]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


??ye!AFi,NWKtYev.DuejuCly,q-Sw&cMgnVwd?hKbfN!peQ'YdtYIlv,iAUCUv,pIcQ&ysBcajx?wGXMpJmp3UV&DW
jKsmOjKOdErWnXaVjuB-GfxKonvi-wNZa:wpsufIfNwNfN
oXQWUvEalgqW
RIBhir
gis?aQJLKl'F$-YQiy,,y&yYEmYnbQcLPdnosflBNxEXEnERp!zGH,pbToUIWOlwNFhsBuD-3OlxBirC-B;qcFlwelv,SPTV&y$JZ-w3:evN:pW z q-wahOXKZN!uYvEJUgCZibRSPlvxc!ujKHRSqffz:CXlboiqJv'bJXe;.Era;.
DyhSH-P,lgbKo?dl&yiMrDOwIBTa!wfROdP!a!BYWpG&pJK&y&'nV&ohpwaLI.P
D$a
' FPda&eAOpbsv
V;h
GffalgDg.q-UzXAcbjRkS-wbr&v?B!hRyhzSJMrV.:wf''MkNEeNIu
vUAJKHF ER:fly?EXjwXlx


## The mathematical trick in self-attention

In [36]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x) # (B, T, 16)
out = wei @ v 
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [40]:
wei.shape, v.shape

(torch.Size([4, 8, 8]), torch.Size([4, 8, 16]))

In [38]:
key

Linear(in_features=32, out_features=16, bias=False)

In [37]:
x

tensor([[[ 0.1808, -0.0700, -0.3596,  ..., -0.8016,  1.5236,  2.5086],
         [-0.6631, -0.2513,  1.0101,  ...,  1.5333,  1.6097, -0.4032],
         [-0.8345,  0.5978, -0.0514,  ..., -0.4370, -1.0012, -0.4094],
         ...,
         [-0.8961,  0.0662, -0.0563,  ...,  2.1382,  0.5114,  1.2191],
         [ 0.1910, -0.3425,  1.7955,  ...,  0.3699, -0.5556, -0.3983],
         [-0.5819, -0.2208,  0.0135,  ..., -1.9079, -0.5276,  1.0807]],

        [[ 0.4562, -1.0917, -0.8207,  ...,  0.0512, -0.6576, -2.5729],
         [ 0.0210,  1.0060, -1.2492,  ...,  0.7859, -1.1501,  1.3132],
         [ 2.2007, -0.2195,  0.5427,  ..., -0.6445,  1.0834, -0.7995],
         ...,
         [ 0.3091,  1.1661, -2.1821,  ...,  0.6151,  0.6763,  0.6228],
         [ 0.0943, -0.3156,  0.7850,  ..., -1.5735,  1.3876,  0.7251],
         [ 0.6455, -0.3313, -1.0390,  ...,  0.0895, -0.3748, -0.4781]],

        [[-0.6067,  1.8328,  0.2931,  ...,  1.0041,  0.8656,  0.1688],
         [-0.2352, -0.2586,  0.0131,  ...,  0

#### End to End Code

In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)
text = open(
    r"C:\Users\Lenovo\Desktop\NanoGPT\Small_Language_Models\data\shakespeare_literature.txt",
    encoding="utf-8"
).read().splitlines()
text = "\n".join(text)
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# with open('input.txt', 'r', encoding='utf-8') as f:
#     text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [4]:
len(chars)

65

In [5]:
model = BigramLanguageModel()

In [10]:
model.token_embedding_table, model.position_embedding_table

(Embedding(65, 64), Embedding(32, 64))

In [8]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.4112, val loss 4.4015
step 4: train loss 3.7900, val loss 3.8028

iotaOr bg&ft ;SbfNl$b; re'yipb'
 he;MfADiMeUh;mORytKEd?$oe nZyrtyhuH ;rcrd tj znip&egYnereHXNR RhtiY
rgPXRrhaefhMwey;fllIR L!er mfozWhtY3 sUg3, uhv!WTh YfQ,gha QXcpfo-wKyV ctQ&eVYiHjchwiUKeeKeOneN hzb.eppneevhecIHVswJs?HstfYte.nnhaKe JbhR  h'ciVXiLenpUteze.e utUvqt.pUKVK
sl y 3qY apC Bztcfn$ rPXeJf y?
' Xsr3
DJh
3ftSfstxt bYB
PE'e$JRjewqle ;aKfs sJEppfsepDhepe, eioci t s?''cttRo l'JbHW s? fwVpr3D;oUzFeXU$R;i& Ov;EsK
i sYwcfMxELEinR&khh!W& LK;&n.z;sQKP ltPrep3cefm'cK&p dpU&S?f CQeci KLeFcc$
UhrhA f jyply dNsBupNW.rRi Y
 K..JhYU iojJHL?Gyiid  em$3 $WfespwY'F
JRze.g.&tuL twqei$rlpbUp AIhYo.e,gymLhfeMHnUcYt kYhtBNZw!DfUrek recrfsHibUPec?uxULYU
piiuoP&l ;lrg OvC VHcR,Uty,b K,e,dKerT HhLtrVctfA'cpzveUte. ctN
Jrrg
&oHfDKXX YtFFKlmoZEpK b'O.
Jrst? SdbiKSriiO z;A:,PUBuKP oo;y;&z3H
JRf Y?L  EgFnt e
cor;'IBoDw$-nhoKh
-z&U
ib  i3, X AcBr;z!z sifRrodvIciHUeu rtcGxnQRLql  y ab KE WEd R zIvRrp

## Using only basic torch

In [ ]:
"""
NanoGPT — raw PyTorch tensors only.

No nn.Module, nn.Linear, nn.Embedding, nn.LayerNorm, nn.Dropout,
nn.Sequential, torch.nn.functional, or any other torch.nn abstractions.

Every weight matrix is a plain torch.Tensor with requires_grad=True.
Every forward pass is written explicitly as tensor math.
Every activation / loss function is implemented from scratch.
Every class is a plain Python class — no inheritance from anything.
"""

import torch
import math

# ────────────────────────── functional primitives ──────────────────────
# Hand-rolled replacements for everything that was in torch.nn.functional.

def softmax(x, dim):
    """
    Numerically stable softmax along `dim`.

    Naive softmax (exp(x) / sum(exp(x))) explodes when x contains large
    values — exp(1000) is inf in float32.

    The standard fix: subtract the max along `dim` before exponentiating.
    This is mathematically identical because the max cancels in numerator
    and denominator, but keeps all exp() inputs <= 0, so no overflow.

        softmax(x)_i = exp(x_i - max(x)) / sum_j( exp(x_j - max(x)) )
    """
    x_max = x.max(dim=dim, keepdim=True).values   # (…, 1) — subtract for stability
    e_x   = torch.exp(x - x_max)                  # (…, vocab) all values <= 0 now
    return e_x / e_x.sum(dim=dim, keepdim=True)   # normalise → probabilities


def cross_entropy(logits, targets):
    """
    Cross-entropy loss for a classification problem.

    logits  : (N, C) — raw unnormalised scores per class
    targets : (N,)   — integer class indices in [0, C)

    Derivation
    ----------
    Cross-entropy = -log( softmax(logits)[target_class] )

    Instead of computing full softmax and then indexing, we use the
    log-sum-exp trick for numerical stability:

        log softmax(x)_i = x_i - log( sum_j exp(x_j) )

    We subtract max(x) before both exponentiation and the outer log
    (the two max terms cancel, leaving the result identical):

        log softmax(x)_i = (x_i - max) - log( sum_j exp(x_j - max) )

    This keeps every exp() input <= 0, avoiding overflow.

    The loss is the mean over the batch.
    """
    N, C = logits.shape

    # Stability shift
    x_max     = logits.max(dim=1, keepdim=True).values       # (N, 1)
    shifted   = logits - x_max                               # (N, C)

    log_sum_exp = torch.log(torch.exp(shifted).sum(dim=1))   # (N,)

    # Pick the logit of the correct class for each sample using
    # advanced indexing:  shifted[row_idx, target_class]
    correct_logits = shifted[torch.arange(N), targets]       # (N,)

    # log-softmax of the correct class
    log_probs = correct_logits - log_sum_exp                  # (N,)

    # Cross-entropy = mean of negative log-probabilities
    return -log_probs.mean()


def relu(x):
    """
    Rectified Linear Unit: max(0, x) element-wise.

    The simplest nonlinearity. Keeps positive values unchanged,
    zeros out negatives. Cheap to compute and backprop through.
    """
    return torch.clamp(x, min=0.0)

# ─────────────────────────── hyperparameters ───────────────────────────

batch_size  = 16    # sequences processed in parallel
block_size  = 32    # maximum context length (sequence length T)
max_iters   = 5000
eval_interval = 500
learning_rate = 1e-3
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters  = 200
n_embd      = 64    # embedding dimension (C)
n_head      = 4     # number of attention heads
n_layer     = 4     # number of transformer blocks
dropout_p   = 0.0   # dropout probability (0 = disabled)

torch.manual_seed(1337)

# ──────────────────────────── data loading ─────────────────────────────

with open("shakespeare_literature.txt", encoding="utf-8") as f:
    text = f.read()

chars      = sorted(set(text))
vocab_size = len(chars)
stoi       = {ch: i for i, ch in enumerate(chars)}
itos       = {i: ch for i, ch in enumerate(chars)}
encode     = lambda s: [stoi[c] for c in s]
decode     = lambda l: ''.join([itos[i] for i in l])

data       = torch.tensor(encode(text), dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split):
    """
    Sample a random batch of (input, target) token sequences.

    Returns
    -------
    x : (B, T)  — input token ids
    y : (B, T)  — target token ids (x shifted right by 1)
    """
    src  = train_data if split == 'train' else val_data
    ix   = torch.randint(len(src) - block_size, (batch_size,))
    x    = torch.stack([src[i : i + block_size]      for i in ix])
    y    = torch.stack([src[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ────────────────────────── weight helpers ─────────────────────────────

def kaiming_uniform(shape, fan_in):
    """
    Kaiming (He) uniform initialisation — the default PyTorch uses for
    nn.Linear weights.  Keeps variance stable through ReLU layers.

    bound = sqrt(6 / fan_in)
    """
    bound = math.sqrt(6.0 / fan_in)
    return torch.empty(shape, device=device).uniform_(-bound, bound)

def zeros(shape):
    return torch.zeros(shape, device=device)

def ones(shape):
    return torch.ones(shape, device=device)

# ─────────────────────────── layer classes ─────────────────────────────

class Linear:
    """
    Fully-connected (affine) layer: out = x @ W.T + b

    Replaces nn.Linear.

    Parameters
    ----------
    in_features  : int
    out_features : int
    bias         : bool — whether to include a bias vector
    """

    def __init__(self, in_features, out_features, bias=True):
        # Weight matrix: (out_features, in_features)
        # We store it transposed relative to the math so the forward
        # pass is just x @ W, mirroring nn.Linear's behaviour.
        self.W = kaiming_uniform((in_features, out_features), fan_in=in_features)
        self.W.requires_grad_(True)

        self.b = None
        if bias:
            self.b = zeros((out_features,))
            self.b.requires_grad_(True)

    def __call__(self, x):
        """
        Forward pass.

        x : (..., in_features)
        returns : (..., out_features)
        """
        out = x @ self.W          # (..., out_features)
        if self.b is not None:
            out = out + self.b    # broadcasts over all leading dims
        return out

    def parameters(self):
        return [self.W] + ([self.b] if self.b is not None else [])


class Embedding:
    """
    Lookup table: maps integer token ids → dense vectors.

    Replaces nn.Embedding.

    Parameters
    ----------
    num_embeddings : int — vocabulary / table size
    embedding_dim  : int — vector width (C)
    """

    def __init__(self, num_embeddings, embedding_dim):
        # Each row is one embedding vector, initialised from N(0,1).
        self.weight = torch.randn(
            (num_embeddings, embedding_dim), device=device
        )
        self.weight.requires_grad_(True)

    def __call__(self, idx):
        """
        idx : (B, T)  integer tensor of ids
        returns : (B, T, embedding_dim)
        """
        return self.weight[idx]   # fancy indexing — differentiable in autograd

    def parameters(self):
        return [self.weight]


class LayerNorm:
    """
    Layer normalisation over the last dimension.

    Replaces nn.LayerNorm.

    Given input x of shape (..., C):
        mean  = mean over C
        var   = variance over C
        x_hat = (x - mean) / sqrt(var + eps)
        out   = gamma * x_hat + beta

    gamma (scale) and beta (shift) are learnable, both initialised to
    1 and 0 respectively — so the layer starts as an identity.
    """

    def __init__(self, ndim, eps=1e-5):
        self.eps   = eps
        self.gamma = ones((ndim,))          # scale — start at 1
        self.beta  = zeros((ndim,))         # shift — start at 0
        self.gamma.requires_grad_(True)
        self.beta.requires_grad_(True)

    def __call__(self, x):
        """
        x : (..., ndim)
        returns : (..., ndim)  — normalised, scaled, shifted
        """
        mean  = x.mean(dim=-1, keepdim=True)            # (..., 1)
        var   = x.var(dim=-1, keepdim=True, unbiased=False)  # (..., 1)
        x_hat = (x - mean) / torch.sqrt(var + self.eps) # (..., ndim)
        return self.gamma * x_hat + self.beta

    def parameters(self):
        return [self.gamma, self.beta]


class Dropout:
    """
    Dropout regularisation.

    Replaces nn.Dropout.

    During training:  randomly zero out elements with probability p,
                      then scale survivors by 1/(1-p) so the expected
                      sum is unchanged (inverted dropout).
    During inference: pass through unchanged.

    Call  layer.train()  /  layer.eval()  to switch modes.
    """

    def __init__(self, p=0.0):
        self.p        = p
        self.training = True   # start in training mode

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    def __call__(self, x):
        if not self.training or self.p == 0.0:
            return x
        # Bernoulli mask: 1 with prob (1-p), 0 with prob p
        mask = (torch.rand_like(x) > self.p).float()
        return x * mask / (1.0 - self.p)   # scale to preserve expectation

    def parameters(self):
        return []   # no learnable weights

# ─────────────────────────── attention head ────────────────────────────

class Head:
    """
    Single causal self-attention head.

    Steps
    -----
    1. Project x into queries Q, keys K, values V via three Linear layers.
    2. Compute raw attention scores:  scores = Q @ K.T / sqrt(head_size)
       Scaling by 1/sqrt(d) keeps the dot-products from growing too large,
       which would push softmax into saturation.
    3. Apply causal mask: positions can only attend to earlier positions
       (upper triangle set to -inf so softmax gives them zero weight).
    4. Softmax → attention weights.
    5. Weighted sum of V:  out = weights @ V
    """

    def __init__(self, head_size):
        self.key   = Linear(n_embd, head_size, bias=False)
        self.query = Linear(n_embd, head_size, bias=False)
        self.value = Linear(n_embd, head_size, bias=False)
        self.dropout = Dropout(dropout_p)

        # Causal mask: lower-triangular matrix of ones.
        # Shape (block_size, block_size) — sliced to (T, T) at runtime.
        # register as a plain tensor (not a parameter — no gradient needed).
        self.tril = torch.tril(
            torch.ones(block_size, block_size, device=device)
        )

    def __call__(self, x):
        """
        x   : (B, T, C)
        out : (B, T, head_size)
        """
        B, T, C = x.shape
        head_size = self.key.W.shape[1]   # out_features of key projection

        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # ── attention scores ──────────────────────────────────────────
        # (B, T, hs) @ (B, hs, T)  →  (B, T, T)
        scale = head_size ** -0.5
        wei   = q @ k.transpose(-2, -1) * scale

        # ── causal mask ───────────────────────────────────────────────
        # zero-entries in tril become -inf so softmax treats them as 0
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # ── softmax over key dimension ────────────────────────────────
        wei = softmax(wei, dim=-1)        # (B, T, T)
        wei = self.dropout(wei)

        # ── aggregate values ──────────────────────────────────────────
        v   = self.value(x)               # (B, T, head_size)
        out = wei @ v                     # (B, T, head_size)
        return out

    def train(self):
        self.dropout.train()

    def eval(self):
        self.dropout.eval()

    def parameters(self):
        return (self.key.parameters() +
                self.query.parameters() +
                self.value.parameters())


# ────────────────────────── multi-head attention ────────────────────────

class MultiHeadAttention:
    """
    Run N attention heads in parallel, then concatenate and project.

    Why multiple heads?
    Each head can learn to attend to different aspects of the context
    (syntax, semantics, coreference, …).  Concatenating gives the full
    picture; the projection blends them.

    out = Dropout( Proj( concat(head_0(x), head_1(x), …) ) )
    """

    def __init__(self, num_heads, head_size):
        self.heads   = [Head(head_size) for _ in range(num_heads)]
        # Projection back to n_embd after concatenation
        self.proj    = Linear(n_embd, n_embd)
        self.dropout = Dropout(dropout_p)

    def __call__(self, x):
        """
        x   : (B, T, C)
        out : (B, T, C)
        """
        # Each head produces (B, T, head_size); cat over last dim → (B, T, C)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

    def train(self):
        for h in self.heads: h.train()
        self.dropout.train()

    def eval(self):
        for h in self.heads: h.eval()
        self.dropout.eval()

    def parameters(self):
        params = self.proj.parameters()
        for h in self.heads:
            params += h.parameters()
        return params


# ──────────────────────────── feed-forward ─────────────────────────────

class FeedForward:
    """
    Position-wise feed-forward network (applied independently per token).

    Architecture:
        Linear(C → 4C)  →  ReLU  →  Linear(4C → C)  →  Dropout

    The 4× expansion gives the model a "working memory" to do more
    computation per token before projecting back.
    """

    def __init__(self, n_embd):
        self.fc1     = Linear(n_embd, 4 * n_embd)
        self.fc2     = Linear(4 * n_embd, n_embd)
        self.dropout = Dropout(dropout_p)

    def __call__(self, x):
        """
        x   : (B, T, C)
        out : (B, T, C)
        """
        x = self.fc1(x)
        x = relu(x)          # ReLU: max(0, x) — defined above
        x = self.fc2(x)
        x = self.dropout(x)
        return x

    def train(self):
        self.dropout.train()

    def eval(self):
        self.dropout.eval()

    def parameters(self):
        return self.fc1.parameters() + self.fc2.parameters()


# ──────────────────────────── transformer block ─────────────────────────

class Block:
    """
    One transformer block = self-attention + feed-forward, each with a
    residual connection and layer norm (Pre-LN formulation).

    Pre-LN (norm before the sublayer) is more stable to train than the
    original Post-LN from "Attention is All You Need".

    x = x + Attention( LayerNorm(x) )
    x = x + FeedForward( LayerNorm(x) )
    """

    def __init__(self, n_embd, n_head):
        head_size = n_embd // n_head         # each head gets an equal slice
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = LayerNorm(n_embd)
        self.ln2  = LayerNorm(n_embd)

    def __call__(self, x):
        """
        x   : (B, T, C)
        out : (B, T, C)
        """
        x = x + self.sa(self.ln1(x))    # residual around attention
        x = x + self.ffwd(self.ln2(x))  # residual around feed-forward
        return x

    def train(self):
        self.sa.train()
        self.ffwd.train()

    def eval(self):
        self.sa.eval()
        self.ffwd.eval()

    def parameters(self):
        return (self.sa.parameters() +
                self.ffwd.parameters() +
                self.ln1.parameters() +
                self.ln2.parameters())


# ──────────────────────────── full GPT model ───────────────────────────

class BigramLanguageModel:
    """
    Character-level GPT (Generatively Pre-trained Transformer).

    Architecture
    ------------
    token_emb  : Embedding(vocab_size, C)        — "what token is this?"
    pos_emb    : Embedding(block_size, C)        — "where in the sequence?"
    blocks     : N × Block                       — transformer stack
    ln_f       : LayerNorm(C)                    — final normalisation
    lm_head    : Linear(C → vocab_size, bias=F)  — logits over vocabulary

    The token and position embeddings are simply summed.  The model
    learns both simultaneously.
    """

    def __init__(self):
        self.token_embedding_table    = Embedding(vocab_size, n_embd)
        self.position_embedding_table = Embedding(block_size, n_embd)
        self.blocks = [Block(n_embd, n_head) for _ in range(n_layer)]
        self.ln_f   = LayerNorm(n_embd)
        self.lm_head = Linear(n_embd, vocab_size, bias=False)

    # ── mode switching ────────────────────────────────────────────────

    def train(self):
        """Put all dropout layers into training mode."""
        for block in self.blocks:
            block.train()

    def eval(self):
        """Put all dropout layers into eval mode (dropout disabled)."""
        for block in self.blocks:
            block.eval()

    # ── parameter collection ──────────────────────────────────────────

    def parameters(self):
        """Return every learnable tensor in the model as a flat list."""
        params = (
            self.token_embedding_table.parameters() +
            self.position_embedding_table.parameters() +
            self.ln_f.parameters() +
            self.lm_head.parameters()
        )
        for block in self.blocks:
            params += block.parameters()
        return params

    # ── forward pass ──────────────────────────────────────────────────

    def __call__(self, idx, targets=None):
        """
        Forward pass.

        Parameters
        ----------
        idx     : (B, T) — input token ids
        targets : (B, T) — ground-truth next-token ids (optional)

        Returns
        -------
        logits : (B, T, vocab_size)  — raw (un-normalised) scores
        loss   : scalar or None
        """
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)         # (B, T, C)
        pos_emb = self.position_embedding_table(          # (T, C)
            torch.arange(T, device=device)
        )
        x = tok_emb + pos_emb     # broadcast: (B,T,C) + (T,C) → (B,T,C)

        for block in self.blocks:
            x = block(x)          # (B, T, C)

        x = self.ln_f(x)          # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Cross-entropy expects (N, C) logits and (N,) targets.
            # Flatten the batch and time dimensions together.
            B, T, C = logits.shape
            logits_2d  = logits.view(B * T, C)   # (B*T, vocab_size)
            targets_1d = targets.view(B * T)      # (B*T,)
            loss = cross_entropy(logits_2d, targets_1d)

        return logits, loss

    # ── autoregressive generation ─────────────────────────────────────

    def generate(self, idx, max_new_tokens):
        """
        Autoregressively sample new tokens one at a time.

        At each step:
          1. Crop context to last block_size tokens (positional embedding
             table only has block_size rows).
          2. Forward pass → logits for the last position.
          3. Softmax → probability distribution over vocab.
          4. Multinomial sample → next token.
          5. Append to sequence and repeat.

        Parameters
        ----------
        idx            : (B, T) — seed token ids
        max_new_tokens : int

        Returns
        -------
        idx : (B, T + max_new_tokens)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]           # keep last T tokens
            logits, _ = self(idx_cond)
            logits     = logits[:, -1, :]             # (B, vocab_size)
            probs      = softmax(logits, dim=-1)      # (B, vocab_size)
            idx_next   = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx        = torch.cat((idx, idx_next), dim=1)        # (B, T+1)
        return idx


# ──────────────────────────── optimiser ────────────────────────────────

class AdamW:
    """
    AdamW optimiser — Adam with decoupled weight decay.

    Adam keeps a running estimate of the first moment (mean gradient)
    and second moment (mean squared gradient) for each parameter.
    It uses these to adapt the learning rate per parameter.

    Bias correction in the first few steps prevents the moments from
    being under-estimated while they warm up from zero.

    Weight decay is applied directly to the weights (not via the
    gradient), which is the "decoupled" part that makes it AdamW.
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999),
                 eps=1e-8, weight_decay=0.0):
        self.params       = params
        self.lr           = lr
        self.beta1, self.beta2 = betas
        self.eps          = eps
        self.weight_decay = weight_decay
        self.t            = 0    # step counter

        # First and second moment buffers, one per parameter tensor
        self.m = [torch.zeros_like(p) for p in params]  # mean
        self.v = [torch.zeros_like(p) for p in params]  # variance

    def zero_grad(self):
        """Zero out gradients before the next backward pass."""
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        """Apply one gradient update to all parameters."""
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad

            # Biased moment estimates
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g * g

            # Bias-corrected estimates
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # Decoupled weight decay (applied before the Adam update)
            if self.weight_decay != 0.0:
                p.data -= self.lr * self.weight_decay * p.data

            # Parameter update
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)


# ──────────────────────────── loss estimation ───────────────────────────

@torch.no_grad()
def estimate_loss(model):
    """
    Evaluate mean loss on train and val splits.

    @torch.no_grad() disables gradient tracking for speed and memory
    efficiency — we only need forward passes here.
    """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y     = get_batch(split)
            _, loss  = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# ──────────────────────────── training loop ─────────────────────────────

model     = BigramLanguageModel()
optimizer = AdamW(model.parameters(), lr=learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

for iteration in range(max_iters):

    # ── periodic evaluation ───────────────────────────────────────────
    if iteration % eval_interval == 0 or iteration == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iteration:5d} | "
              f"train loss {losses['train']:.4f} | "
              f"val loss {losses['val']:.4f}")

    # ── forward + backward + update ───────────────────────────────────
    xb, yb        = get_batch('train')
    logits, loss  = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()     # PyTorch autograd computes all gradients for us
    optimizer.step()

# ──────────────────────────── generation ────────────────────────────────

context = torch.zeros((1, 1), dtype=torch.long, device=device)
output  = model.generate(context, max_new_tokens=2000)
print(decode(output[0].tolist()))